<a href="https://colab.research.google.com/github/prithwis/Centaur/blob/main/MOS_Simulation_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![alt text](https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/images/CentaurBanner2.png)<br>


<hr>

[Prithwis Mukerjee](http://www.linkedin.com/in/prithwis)<br>

## Middle East Oil Security Simulation Engine
MOS is an LLM-driven geopolitical simulation in which strategic actors choose actions within an explicitly represented and evolving WorldState. Chanakya proposes actor-specific strategic options, Centaur evaluates their relative likelihood, and ZeitWorld translates the selected Event into the next WorldState. The objective is not to predict the future, but to explore plausible, path-dependent trajectories arising from alternative strategic choices.

In [1]:
!wget -O mos.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mos.py
!wget -O mosZeitWorld.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosZeitWorld.py
!wget -O mosChanakya.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosChanakya.py
!wget -O mosCentaur.py -q https://raw.githubusercontent.com/prithwis/Centaur/refs/heads/main/mosUtils/mosCentaur.py

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *

In [2]:
import importlib

import mos
import mosZeitWorld
import mosChanakya
import mosCentaur

importlib.reload(mos)
importlib.reload(mosZeitWorld)
importlib.reload(mosChanakya)
importlib.reload(mosCentaur)

from mos import *
from mosZeitWorld import *
from mosChanakya import *
from mosCentaur import *

In [3]:
client = authenticateOpenAI()
model = "gpt-5.6-luna"

OpenAI authentication successful ✔
Logged in as Calcutta prithwis@yantrajaal.com


## ZeitWorld - when Events change Worlds

In [4]:
# These are the eight variables that currently define the World
# This number can be increased if necessasary
#
# ---------------------------------------------------------------

worldVariables = [

    WorldVariable(
        name="Oil Price",
        description=(
            "International crude oil price level, represented "
            "by the Brent benchmark."
        ),
        direction="Higher index means higher oil price."
    ),

    WorldVariable(
        name="Hormuz Oil Flow",
        description=(
            "Volume of crude oil and petroleum products successfully "
            "transiting the Strait of Hormuz."
        ),
        direction="Higher index means greater oil flow."
    ),

    WorldVariable(
        name="India Gulf Oil Imports",
        description=(
            "Volume of crude oil imported by India from "
            "Gulf-origin oil-producing countries."
        ),
        direction=(
            "Higher index means greater Indian imports "
            "of Gulf-origin oil."
        )
    ),

    WorldVariable(
        name="Iran Sanctions Enforcement",
        description=(
            "Degree to which restrictions on Iranian petroleum exports, "
            "trade and financial transactions are effectively enforced "
            "rather than circumvented."
        ),
        direction=(
            "Higher index means more effective sanctions enforcement "
            "and less successful circumvention."
        )
    ),

    WorldVariable(
        name="US/Allied Cohesion",
        description=(
            "Degree of alignment between the United States and cooperating "
            "states in their response to Iran and the Gulf crisis."
        ),
        direction="Higher index means greater US/allied cohesion."
    ),

    WorldVariable(
        name="Iran Internal Cohesion",
        description=(
            "Degree of alignment within Iran's political, military and "
            "security establishment in responding to the crisis."
        ),
        direction="Higher index means greater internal Iranian cohesion."
    ),

    WorldVariable(
        name="US-Iran Tension",
        description=(
            "Intensity of strategic hostility between the United States "
            "and Iran across diplomatic, economic and military domains."
        ),
        direction="Higher index means greater US-Iran tension."
    ),

    WorldVariable(
        name="Regional Armed Conflict",
        description=(
            "Intensity and geographic breadth of kinetic military activity "
            "associated with the regional crisis."
        ),
        direction=(
            "Higher index means more intense or geographically "
            "widespread armed conflict."
        )
    )
]

In [5]:
# Historical Data and
# LLM Roles and prompts are stored and retrieved from Google Drive
#
# ----------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

MOS_HOME = "/content/drive/MyDrive/Praxis/Course Geopolitics/Centaur Project/MOS/"
WORLDFACTS_DIR = f"{MOS_HOME}/WorldFacts"
ZW_PATH = f"{MOS_HOME}/ZeitWorld"
CH_PATH = f"{MOS_HOME}/Chanakya"
CT_PATH = f"{MOS_HOME}/Centaur"

!ls -l "{WORLDFACTS_DIR}"
!ls -lh "{WORLDFACTS_DIR}/MD"
!ls -lh "{WORLDFACTS_DIR}/NEWS"
!ls -lh "{ZW_PATH}"
!ls -lh "{CH_PATH}"
!ls -lh "{CT_PATH}"

Mounted at /content/drive
total 12
drwx------ 2 root root 4096 Sep 28 09:09 MD
drwx------ 2 root root 4096 Sep 29 09:42 NEWS
drwx------ 2 root root 4096 Sep 28 06:55 PDF
total 163K
-rw------- 1 root root 7.5K Oct  2 07:03 IranData_261002.md
-rw------- 1 root root 5.6K Oct  2 07:03 MissilesofIranMissil_261002.md
-rw------- 1 root root 8.3K Oct  2 07:04 OilMarketReportSepte_261002.md
-rw------- 1 root root  12K Oct  2 07:04 PressReleasePagePres_261002.md
-rw------- 1 root root 4.3K Oct  2 07:04 ShortTermEnergyOutlo_261002.md
-rw------- 1 root root  37K Oct  2 07:06 SnapshotofIndiasOilG_261002.md
-rw------- 1 root root  18K Oct  2 07:06 UKCommonsIranin2026S_261002.md
-rw------- 1 root root  70K Oct  2 07:08 USCongressR488873_261002.md
total 71K
-rw------- 1 root root 71K Oct  2 07:14 MOS-MC-LiveFeed_261002_1244_IST.txt
total 13K
-rw------- 1 root root 2.2K Oct  1 00:10 ZW_Initialisation_Prompt_v0.txt
-rw------- 1 root root 3.1K Oct  1 14:08 ZW_Initialisation_Prompt_v1.txt
-rw------- 1 roo

In [6]:
# Knowledge of the initial world state is collected from a
# a set of PDF files that have been converted to markdown format
# live news data from 1 source, viz. MoneyControl.com
# Assembled into one large corpus
#
# ------------------------------------------------

WorldFactBase = getWorldFactBase(WORLDFACTS_DIR)


Static sources : 8
Live sources   : 1
Characters     : 235,072
Words          : 35,600


In [7]:
# Zeitworld Roles and Prompts are retrieved from Google Drive

ZW_Role = getPrompt(f"{ZW_PATH}/ZW_Role_v0.txt")

#ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v0.txt")
ZW_Initialisation_Prompt = getPrompt(f"{ZW_PATH}/ZW_Initialisation_Prompt_v1.txt")

ZW_Update_Prompt = getPrompt(f"{ZW_PATH}/ZW_Update_Prompt_v0.txt")

In [8]:
# ZeitWorld object is instantiated from its class along with its ROLE
#
ZeitWorld = ZeitWorld_Agent(client, model, ZW_Role)

#----------
print(ZeitWorld.initialised)
#print(ZeitWorld.model)

False


In [9]:
# W is a list of evolving World States
# An empty list is created. A blank world template is created, without any information
# ----------------------------

W = []

W0_Template = WorldState(
    state_id="W0",
    variables=worldVariables
)

In [10]:
# ZeitWorld reads WorldFactBase data corpus and creates a World State based on World Template
# This initial World State is inserted into the World List W and becomes W[0]
#
# ----------------------------------------

_ = ZeitWorld.initialise(W,W0_Template, WorldFactBase, ZW_Initialisation_Prompt)

print(ZeitWorld.initialised)

Initialising WorldState ... 
Model: gpt-5.6-luna | Effort: medium | Input: 57,064 | Output: 2,044 | Reasoning: 153 | Total: 59,108
True


In [ ]:
# A first look at the intial World State W[0]
# The initial values are all set to 100, as the baseline
#

#W[0].show(-2)  # ---- just the world vectors
#W[0].show()    # ---- the entire state
#W[0].show(4)    # ---- details of any one vector

## Epoch, Events and Chanakya

In [11]:
# ============================================================
# Load Chanakya Roles
# ============================================================

#-rw------- 1 root root 2.0K Oct  2 00:06 CH_Role_Generic_v0.txt
#-rw------- 1 root root 1.8K Oct  2 00:07 CH_Role_Iran_v0.txt
#-rw------- 1 root root 1.8K Oct  2 00:07 CH_Role_US_v0.txt

CH_Generic_Role = getPrompt(f"{CH_PATH}/CH_Role_Generic_v0.txt")
CH_US_Role      = getPrompt(f"{CH_PATH}/CH_Role_US_v0.txt")
CH_IR_Role      = getPrompt(f"{CH_PATH}/CH_Role_Iran_v0.txt")
CH_BinaryHD     = getPrompt(f"{CH_PATH}/CH_Strategy_BinaryHD_v0.txt")

In [12]:
# Instantiate Chanakya objects with their own
# National Actor/National interests
# Strategy -- currently only Binary Hawk / Dove

Chanakya_US = Chanakya_Agent(
    agent_id="ChanakyaUS_HD",
    actor="US",
    generic_role=CH_Generic_Role,
    specific_role=CH_US_Role,
    strategy=CH_BinaryHD
)

Chanakya_US.show()

Chanakya_IR = Chanakya_Agent(
    agent_id="ChanakyaIR_HD",
    actor="Iran",
    generic_role=CH_Generic_Role,
    specific_role=CH_IR_Role,
    strategy=CH_BinaryHD
)

Chanakya_IR.show()

ChanakyaRegistry = {
    "US": Chanakya_US,
    "Iran": Chanakya_IR
}

Chanakya : ChanakyaUS_HD
Actor    : US
Chanakya : ChanakyaIR_HD
Actor    : Iran


### Enter the Centaur

In [13]:
# ============================================================
# Load Centaur Roles
# ============================================================

#-rw------- 1 root root 1.5K Oct  2 14:25 CT_Generic_Role_v0.txt
#-rw------- 1 root root 1.6K Oct  2 14:26 CT_UNSG_Role_v0.txt

CT_Generic_Role = getPrompt(f"{CT_PATH}/CT_Generic_Role_v0.txt")
CT_UNSG_Role      = getPrompt(f"{CT_PATH}/CT_UNSG_Role_v0.txt")

In [14]:
Centaur_UNSG = Centaur_Agent(
    agent_id="CentaurUNSG",
    generic_role=CT_Generic_Role,
    adjudication_role=CT_UNSG_Role
)

In [16]:
# Two optional Triggers, we manually choose any ONE, to start the process

Trigger1 = Event(
    actor="Iran",
    action=(
        "Iran declares the Strait of Hormuz closed to all commercial shipping, "
        "orders its forces to prevent tanker transit, and launches attacks against "
        "US naval forces operating in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


Trigger2 = Event(
    actor="US",
    action=(
        "The United States announces an immediate suspension of offensive military "
        "operations against Iran and offers direct negotiations on maritime security "
        "and sanctions relief, while maintaining defensive protection of US forces "
        "and commercial shipping in the Gulf."
    ),
    rationale = "Initial Trigger for Iran"
)


In [18]:
# Choose the starting trigger event for initial epoch
DefaultTrigger = Trigger1
#Trigger = Trigger2
DefaultTrigger.show()
print(DefaultTrigger.selection_weight)


Event     : 
Actor     : Iran
Action    : Iran declares the Strait of Hormuz closed to all commercial shipping, orders its forces to prevent tanker transit, and launches attacks against US naval forces operating in the Gulf.
Rationale : Initial Trigger for Iran
None


In [19]:
# E is a list of events that will define History
#
E = []

### Epoch - this will loop

In [33]:
# ============================================================
# Establish Current Trigger
# ============================================================

epoch = len(W) - 1
if len(E) == 0:
    CurrentTrigger = DefaultTrigger
    CurrentTrigger.event_id = f"E{epoch}"
    E.append(CurrentTrigger)
else:
    CurrentTrigger = E[-1]

E[epoch].show()

Event     : E2
Actor     : US
Action    : Conduct a coordinated military campaign with US and willing partners to suppress Iranian forces threatening the Strait, strike relevant missile, naval, and command infrastructure, and enforce escorted commercial transit through the waterway.
Rationale : Iran has directly attacked US naval forces and is attempting to close a vital international chokepoint. Imposing immediate military costs and restoring escorted passage could reinforce deterrence, protect US forces and shipping, and prevent Iran from gaining strategic leverage through coercive closure.


In [34]:
print("epoch :", epoch)
print("len(W):", len(W))
print("len(E):", len(E))

epoch : 1
len(W): 2
len(E): 3


In [35]:
# ZeitWorld updates the world [W], moving it from W[epoch] to W[epoch+1] using Trigger Event E[epoch]
#
# -------------------------------

_ = ZeitWorld.update( W,E[epoch],ZW_Update_Prompt)

E[epoch].show()
W[epoch+1].show()

Updating WorldState ---------- W1
Model: gpt-5.6-luna | Effort: medium | Input: 3,374 | Output: 2,373 | Reasoning: 131 | Total: 5,747
Appended  WorldState ---------- W2
Event     : E2
Actor     : US
Action    : Conduct a coordinated military campaign with US and willing partners to suppress Iranian forces threatening the Strait, strike relevant missile, naval, and command infrastructure, and enforce escorted commercial transit through the waterway.
Rationale : Iran has directly attacked US naval forces and is attempting to close a vital international chokepoint. Imposing immediate military costs and restoring escorted passage could reinforce deterrence, protect US forces and shipping, and prevent Iran from gaining strategic leverage through coercive closure.

World State : W2
-------------------------------------------------------
 0  Oil Price                          113.00
 1  Hormuz Oil Flow                     72.00
 2  India Gulf Oil Imports              94.00
 3  Iran Sanctions 

In [36]:
# ============================================================
# Turn Controller
# Who will act next ? Which Actor/Nation ?
# ============================================================

#ActiveChanakya = Chanakya_US
#ActiveChanakya = getActiveChanakya()
ActiveChanakya = getActiveChanakya(E[-1], ChanakyaRegistry)

ActiveChanakya.show()

Chanakya : ChanakyaIR_HD
Actor    : Iran


In [37]:
print(epoch)
E[epoch].show()
W[epoch+1].show()

1
Event     : E2
Actor     : US
Action    : Conduct a coordinated military campaign with US and willing partners to suppress Iranian forces threatening the Strait, strike relevant missile, naval, and command infrastructure, and enforce escorted commercial transit through the waterway.
Rationale : Iran has directly attacked US naval forces and is attempting to close a vital international chokepoint. Imposing immediate military costs and restoring escorted passage could reinforce deterrence, protect US forces and shipping, and prevent Iran from gaining strategic leverage through coercive closure.

World State : W2
-------------------------------------------------------
 0  Oil Price                          113.00
 1  Hormuz Oil Flow                     72.00
 2  India Gulf Oil Imports              94.00
 3  Iran Sanctions Enforcement         100.00
 4  US/Allied Cohesion                 108.00
 5  Iran Internal Cohesion             100.00
 6  US-Iran Tension                    121.00
 7

In [38]:
print("epoch :", epoch)
print("len(W):", len(W))
print("len(E):", len(E))

epoch : 1
len(W): 3
len(E): 3


In [39]:
# ============================================================
# Generate Candidate Reaction Events
# ============================================================

Candidates = ActiveChanakya.propose(
    client=client,
    model=model,
    trigger=E[epoch],
    world=W[epoch+1]
)

Chanakya working ---------- ChanakyaIR_HD
Model: gpt-5.6-luna | Effort: medium | Input: 2,440 | Output: 366 | Reasoning: 108 | Total: 2,806


In [40]:
for i, candidate in enumerate(Candidates):
    print(f"\nCandidate Reaction {i+1}")
    print("-" * 60)
    candidate.show()


Candidate Reaction 1
------------------------------------------------------------
Event     : 
Actor     : Iran
Action    : HAWK: Intensify calibrated asymmetric resistance to the escorted transit campaign by dispersing surviving missile and naval assets, conducting further attacks on military escorts and supporting bases, and activating aligned forces against US and partner interests across regional theaters.
Rationale : Iran could seek to raise the operational and political cost of enforcing transit, preserve deterrence after direct strikes on its infrastructure, and demonstrate that military pressure cannot secure the Strait without continued risk. Dispersal and geographically distributed pressure could help retain leverage despite the campaign against Iranian command and maritime assets.

Candidate Reaction 2
------------------------------------------------------------
Event     : 
Actor     : Iran
Action    : DOVE: Offer an immediate, conditional de-escalation through intermediar

In [41]:
# ============================================================
# Centaur Adjudication
# ============================================================

weights = Centaur_UNSG.evaluate(
    client,
    model,
    CurrentTrigger,
    W[epoch + 1],
    Candidates
)

print("\nCentaur Weights")
print("----------------")
print(weights)

Centaur working ---------- CentaurUNSG
Model: gpt-5.6-luna | Effort: medium | Input: 2,338 | Output: 83 | Reasoning: 67 | Total: 2,421

Centaur Weights
----------------
[0.72, 0.28]


In [ ]:
#print(Chanakya_US._event_to_text(E[t]))
#print()
#print(Chanakya_US._world_to_text(W[t+1]))

In [ ]:
'''
Reaction = Candidates[0]     # manual selection for now
Reaction.event_id = f"E{len(E)}"
E.append(Reaction)
'''

In [42]:
Reaction = Shakuni(Candidates, weights)

Reaction.event_id = f"E{len(E)}"
E.append(Reaction)

print("\nShakuni has spoken")
print("------------------")
Reaction.show()
print(Reaction.selection_weight)


Shakuni has spoken
------------------
Event     : E3
Actor     : Iran
Action    : DOVE: Offer an immediate, conditional de-escalation through intermediaries, including a pause in attacks on escorted shipping and US forces in exchange for a reciprocal halt to strikes and blockade operations, followed by negotiations on sanctions relief, nuclear issues, and the status of commercial transit.
Rationale : Iran could use the existing intermittent diplomatic channels to reduce the danger of sustained attacks and further damage to its military and economic position while avoiding unconditional capitulation. A conditional offer would preserve bargaining leverage, create space for internal consensus, and frame reopening the Strait as part of a reciprocal settlement rather than a unilateral concession.
0.28


In [43]:
print(Reaction.event_id)
print(Reaction.actor)
print(Reaction.action)
print(Reaction.rationale)

E3
Iran
DOVE: Offer an immediate, conditional de-escalation through intermediaries, including a pause in attacks on escorted shipping and US forces in exchange for a reciprocal halt to strikes and blockade operations, followed by negotiations on sanctions relief, nuclear issues, and the status of commercial transit.
Iran could use the existing intermittent diplomatic channels to reduce the danger of sustained attacks and further damage to its military and economic position while avoiding unconditional capitulation. A conditional offer would preserve bargaining leverage, create space for internal consensus, and frame reopening the Strait as part of a reciprocal settlement rather than a unilateral concession.


In [32]:
print("epoch :", epoch)
print("len(W):", len(W))
print("len(E):", len(E))

epoch : 0
len(W): 2
len(E): 3


### Epoch Loop Ends

In [ ]:
def resetWorld():
    global ActiveChanakya, Candidates
    del W[1:]
    E.clear()
    ActiveChanakya = None
    Candidates = []

    print("World reset to W0.")
# ------------------------
resetWorld()

World reset to W0.


#ChronoTantra | ChronoYantra | ChronoMantra | ChronoSutra
 [More Information ](http://bit.ly/chronobooks) <br>

[![More Information](https://raw.githubusercontent.com/prithwis/parashar21/refs/heads/main/images/CTYMS-Colab-Footer.png)](https://chronos.yantrajaal.com)